GSEA中 enrichmentScore NES p.adjust qvalue rank leading_edge 分别什么意思， p.adjust qvalue 中哪一个是FDR， 哪一个更strict？

在 GSEA（基因集富集分析）的结果表格中（尤其是使用 R 包 clusterProfiler 运行时），这些指标共同决定了一个通路（Gene Set）是否真正被显著激活或抑制。

下面为您逐一拆解这些指标的真实含义，并重点解答您关于 FDR 和 严格性 的疑问。

一、 GSEA 核心指标含义详解


| 指标名称 | 英文全称 | 核心含义（通俗解释） |
| :--- | :--- | :--- |
| **`enrichmentScore` (ES)** | Enrichment Score | **富集评分**。反映该通路基因在全基因组排序列表两端的富集程度。正数表示倾向于在列表顶部（上调），负数表示倾向于在底部（下调）。 |
| **`NES`** | Normalized Enrichment Score | **标准归一化富集评分**。因为不同通路的基因数目不同，无法直接比较 ES。NES 是根据通路大小进行校正后的得分，**是跨通路比较、筛选核心通路的最重要指标**（通常 $\lvert \text{NES} \rvert > 1$ 或 $> 1.5$ 认为有意义）。 |
| **`p.adjust`** | Adjusted p-value | **校正后的 $p$ 值**。为了控制多重假设检验的假阳性，默认使用 **BH (Benjamini-Hochberg) 法** 对名义 $p$ 值（pvalue）进行修正后的结果。 |
| **`qvalue`** | q-value | **$q$ 值**。通常采用 John Storey 开发的算法计算，用于估算假阳性比例。 |
| **`rank`** | Rank at Max | **峰值位置（排位）**。指当 ES 评分达到最高峰（Peak）时，对应的那只基因在整个排序列表中的具体序号（位置）。 |
| **`leading_edge`** | Leading Edge | **核心富集统计指标**。通常包含三个子参数（如 `tags=XX%, list=YY%, signal=ZZ%`），用于定量描述“核心基因（Leading edge genes）”的特征。 |

补充：如何理解 leading_edge 中的三个指标？
tags：在该通路所有基因中，对贡献了富集得分峰值（即排在 rank 之前）的基因所占的比例。

list：这部分核心基因占整个基因组排序列表（通常有一两万个基因）的比例。

signal：通过公式综合 tags 和 list 计算出的信号强度。如果该通路很显著，signal 值通常会比较高。

二、 p.adjust 和 qvalue 中，哪一个是 FDR？

两个都是用来控制 FDR（False Discovery Rate，错误发现率）的方法，但在学术界和具体软件里，通常 qvalue 更对标 Broad Institute 官方 GSEA 定义的 "FDR q-val"。

p.adjust：使用经典的 Benjamini-Hochberg (BH) 算法。它直接对 $p$ 值进行放大修正。

qvalue：使用 Storey 算法。它在经典 BH 法的基础上，引入了一个敏感参数 $\pi_0$（代表真正属于无效假设/无差异的通路比例）。

三、 p.adjust 和 qvalue 哪一个更严格（Strict）？

p.adjust 比 qvalue 更严格（更保守）。🧠 为什么 p.adjust 更严格？
从数学原理上看，Storey 的 qvalue 计算公式可以通俗地理解为：$$\text{qvalue} = \text{BH-adjusted } p\text{-value} \times \pi_0$$

因为 $\pi_0$（无差异通路比例）永远 $\le 1$（通常在 $0.4 \sim 0.8$ 之间），

所以在同一个分析结果中，qvalue 的值永远小于或等于 p.adjust 的值（即 $\text{qvalue} \le \text{p.adjust}$）。值越小越容易显著。这意味着，如果你设定相同的阈值（比如 $0.05$）：用 p.adjust < 0.05 筛选出来的通路会比较少（条件苛刻，假阳性极低，但容易漏掉潜在通路）。

用 qvalue < 0.05 筛选出来的通路会比较多（条件相对宽松，统计学功效更高）。

💡 实际生信分析中的过滤建议在撰写论文和过滤 GSEA 结果时，

业界公认的常规筛选标准通常满足以下其一即可：

官方推荐标准：NES > 1.5（或 $\lvert \text{NES} \rvert > 1$） 且 qvalue < 0.25（GSEA 官方认为对于探索性研究，FDR 设为 25% 是可接受的）。

严格发表标准：NES > 1.5 且 p.adjust < 0.05（如果你的差异非常明显，用 p.adjust 过滤出的通路更经得起推敲）。